# Notebook 7 — Audit manuel des étiquettes MIMIC (CPU)

**Réglages :** *Accelerator* **None**. *Inputs* : le dataset du code et la sortie du notebook 1 (cache).

1. La cellule 2 tire 200 comptes rendus (échantillon stratifié par pathologie) et écrit dans
   `/kaggle/working/audit/` :
   - `grille_audit_mimic.xlsx` : la grille à remplir (onglet **guide** = consignes, onglet **grille** = à annoter) ;
   - `audit_key.csv` : les étiquettes des règles, **à ne pas ouvrir avant d'avoir annoté** (évaluation en aveugle).
2. Téléchargez les deux fichiers (panneau *Output*), remplissez la grille dans Excel ou LibreOffice
   (environ 45 min), puis envoyez-moi la grille remplie **et** `audit_key.csv` : je calcule les résultats.
3. (Facultatif) Cellule 3 : calcul sur Kaggle si vous ajoutez la grille remplie comme dataset.

In [ ]:
import glob, os, re, shutil
def _version(init):
    m = re.search(r'__version__\s*=\s*"([^"]+)"', open(init).read())
    return m.group(1) if m else "0"
# plusieurs copies du code peuvent être présentes (dataset du code, sorties d'anciens notebooks) :
# on prend la plus récente
hits = sorted(glob.glob("/kaggle/input/**/ecgfusion/__init__.py", recursive=True), key=_version)
assert hits, "Code introuvable : ajoutez le dataset contenant ecg_fusion.zip via « Add Input »."
src = os.path.dirname(os.path.dirname(hits[-1]))
shutil.rmtree("/tmp/ecg_fusion", ignore_errors=True)
shutil.copytree(src, "/tmp/ecg_fusion")   # hors /kaggle/working : le code n'encombre pas la sortie
os.chdir("/tmp/ecg_fusion")
!pip install -q wfdb
print(f"Code version {_version(hits[-1])} prêt (source : {src})")

In [ ]:
metas = sorted(glob.glob("/kaggle/input/**/mimic/meta.csv", recursive=True))
assert metas, "meta.csv MIMIC introuvable : ajoutez la sortie du notebook 1 en Input."
CACHE = os.path.dirname(metas[0]); print("Cache MIMIC :", CACHE)
!python scripts/audit_sample.py --cache {CACHE} --out /kaggle/working/audit

In [ ]:
# Facultatif : calcul des résultats sur Kaggle (grille remplie ajoutée en Input, nom contenant "rempli")
filled = sorted(glob.glob("/kaggle/input/**/*rempli*.xlsx", recursive=True))
if filled:
    !python scripts/audit_score.py --grid {filled[0]} --key /kaggle/working/audit/audit_key.csv --out /kaggle/working/audit
else:
    print("Pas de grille remplie en Input : envoyez-la-moi avec audit_key.csv.")